# Phase 2 — Chronological Split and Cross-Validation Folds

**Purpose.** Split the monthly data in time, and define the validation folds used for every modeling decision. Both steps run in SQL:

- `sql/003_phase2_split.sql` assigns every month of `county_month_joined` to Train (through 2021-09-30), Embargo (2021-Q4, excluded from both sides) or Test (2022-01-01 to 2026-03-31). Months after that are labelled `unassigned` and never dropped silently.
- `sql/003b_cv_folds.sql` generates the 11 expanding-window folds (validation years 2010 to 2020) in a table, and stops with an error if the design is inconsistent.

Input: `county_month_joined`, built by notebook 01. Nothing is saved to files: later phases read the tables straight from Postgres.

**Why the split looks like this.** The Test window contains the 2022-23 rate-hiking cycle, while the two earlier cycles (2004-06 and 2015-18) are in Train, so the models are tested on a hiking episode they never trained on. The one-quarter embargo keeps the last Train months, whose momentum features overlap in time, away from the first Test months.

## 1. Setup and connection

In [1]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv(override=True)

DB_USER = os.environ.get("PGUSER", "postgres")
DB_PASSWORD = os.environ.get("PGPASSWORD", "")
DB_HOST = os.environ.get("PGHOST", "localhost")
DB_PORT = os.environ.get("PGPORT", "5432")
DB_NAME = os.environ.get("PGDATABASE", "housing_risk_sql")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


def run_sql_file(name):
    """Run a whole SQL file through the raw database cursor, with no parameters.

    Splitting a file on ';' breaks on semicolons inside comments and inside DO blocks,
    so the file is sent as one piece instead. Notices raised by the file are printed.
    """
    path = Path("../sql") / name
    raw = engine.raw_connection()
    try:
        cur = raw.cursor()
        cur.execute(path.read_text(encoding="utf-8"))
        raw.commit()
        for notice in getattr(raw, "notices", []):
            print(notice.strip())
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()
    print(f"ran {path}")


with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database();")).scalar())
    n_joined = conn.execute(text("SELECT COUNT(*) FROM county_month_joined")).scalar()
print(f"county_month_joined: {n_joined:,} rows")

capstone_sql_variant
county_month_joined: 709,996 rows


## 2. Run the split

Rebuilds `monthly_split` (dropped first, so this is safe to re-run) and its three per-window views. The file ends with a count per split, which is read back below.

In [2]:
run_sql_file("003_phase2_split.sql")

split_counts = pd.read_sql(
    """
    SELECT split, COUNT(*) AS n_rows, COUNT(DISTINCT county_fips) AS n_counties,
           MIN(month) AS first_month, MAX(month) AS last_month
    FROM monthly_split GROUP BY split ORDER BY split
    """, engine)
print(f"Total: {split_counts['n_rows'].sum():,} rows")
split_counts

NOTICE:  drop cascades to 3 other objects
DETAIL:  drop cascades to view train_monthly
drop cascades to view embargo_monthly
drop cascades to view test_monthly
ran ..\sql\003_phase2_split.sql
Total: 709,996 rows


,split,n_rows,n_counties,first_month,last_month
0,embargo,9063,3021,2021-10-31,2021-12-31
1,test,156364,3071,2022-01-31,2026-03-31
2,train,529214,3021,2000-01-31,2021-09-30
3,unassigned,15355,3071,2026-04-30,2026-08-31


**Expected with the Zillow file downloaded on 2026-09-23:** embargo 9,063, test 156,364, train 529,214, unassigned 15,355, which is 709,996 in total, including the 1,043 empty internal-gap months. The unassigned bucket is 5 months x 3,071 counties = 15,355. The figures drift slightly as Zillow publishes new months.

## 3. Leakage check

The three windows are mutually exclusive by construction (one `CASE` expression), so a row can never land in two splits. This check confirms that the `CASE` conditions have no gaps, that no row was lost, and that the windows follow each other in time.

In [3]:
with engine.connect() as conn:
    null_split = conn.execute(text("SELECT COUNT(*) FROM monthly_split WHERE split IS NULL")).scalar()

total = int(split_counts["n_rows"].sum())
by = split_counts.set_index("split")
print(f"Rows in monthly_split: {total:,} | rows without a split label: {null_split:,}")

assert null_split == 0, "Some rows got no split label: a CASE condition has a gap."
assert total == n_joined, "The split lost or duplicated rows."
assert by.loc["train", "last_month"] < by.loc["embargo", "first_month"], "Train and Embargo overlap."
assert by.loc["embargo", "last_month"] < by.loc["test", "first_month"], "Embargo and Test overlap."
print("Every month has exactly one split, and Train < Embargo < Test in time.")

Rows in monthly_split: 709,996 | rows without a split label: 0
Every month has exactly one split, and Train < Embargo < Test in time.


## 4. Cross-validation folds, generated in SQL

Every modeling choice is made on expanding-window folds inside Train:

- fold 0 trains on 2000-Q1 to 2009-Q4 (40 quarters) and validates on 2010 (4 quarters)
- each later fold adds the previous validation year to its training window
- 11 folds in total, validation years 2010 to 2020, all inside Train

`sql/003b_cv_folds.sql` generates them with `generate_series` and checks them: the expected number of folds, validation windows inside Train, validation starting the quarter after training ends, exactly 4 quarters per validation window, and a clean expanding window. Any failure stops the run with an error.

In [4]:
run_sql_file("003b_cv_folds.sql")

folds = pd.read_sql("SELECT * FROM cv_folds ORDER BY fold_id", engine)
folds

NOTICE:  drop cascades to view cv_fold_rows
NOTICE:  Fold checks passed: 11 folds, validation years 2010 to 2020, last Train month 2021-09-30
ran ..\sql\003b_cv_folds.sql


,fold_id,train_start,train_end,val_start,val_end,n_train_quarters,n_val_quarters
0,0,2000-03-31,2009-12-31,2010-03-31,2010-12-31,40,4
1,1,2000-03-31,2010-12-31,2011-03-31,2011-12-31,44,4
2,2,2000-03-31,2011-12-31,2012-03-31,2012-12-31,48,4
3,3,2000-03-31,2012-12-31,2013-03-31,2013-12-31,52,4
4,4,2000-03-31,2013-12-31,2014-03-31,2014-12-31,56,4
5,5,2000-03-31,2014-12-31,2015-03-31,2015-12-31,60,4
6,6,2000-03-31,2015-12-31,2016-03-31,2016-12-31,64,4
7,7,2000-03-31,2016-12-31,2017-03-31,2017-12-31,68,4
8,8,2000-03-31,2017-12-31,2018-03-31,2018-12-31,72,4
9,9,2000-03-31,2018-12-31,2019-03-31,2019-12-31,76,4


**How the folds are used**

- Rows are assigned to a fold by their **label quarter**, the quarter whose decline they describe, with features measured at the end of the quarter before. The membership view `cv_fold_rows` (`sql/005b_cv_fold_rows.sql`) needs the modeling table, so it is built and checked in notebook 04.
- **No gap between training and validation.** Training ends the quarter before validation starts. Every training label is already known when the validation features are measured, and the features only look backward, so this is how a model would be used in practice. A gap would make the scores more pessimistic than real use. The Train/Test embargo is stricter on purpose, because no choice may be made from Test.
- **2021-Q1 to Q3 are in Train but in no fold.** They are used only when the final models are refitted on all of Train.
- **Positives per fold.** Every validation fold needs at least 20 decline cases before its AUC is trusted. That gate is checked in notebook 04 and in `sql/006_checks.sql`.

## Status

Phase 2 is complete when the split counts match the expected figures and the fold checks pass. The next notebooks read the tables straight from Postgres, a live view instead of a file snapshot.